In [ ]:
!pip install torch numpy scipy matplotlib

In [ ]:
%%writefile dataset_generation.py
import numpy as np
from scipy.stats import qmc

def latin_hypercube_sample(n_samples, n_dims, low, high, seed=42):
    sampler = qmc.LatinHypercube(d=n_dims, seed=seed)
    sample = sampler.random(n=n_samples)
    return qmc.scale(sample, low, high)

def toy1_function(X):
    x, y, t, z = X[:, 0], X[:, 1], X[:, 2], X[:, 3]
    return (np.exp(-0.5 * x)
            + np.log(1 + np.exp(0.4 * y))
            + np.tanh(t)
            + np.sin(z)
            - 0.4)

def generate_toy1():
    X_train = latin_hypercube_sample(500,  4, low=0.0, high=4.0, seed=42)
    y_train = toy1_function(X_train)
    X_test  = latin_hypercube_sample(5000, 4, low=0.0, high=6.0, seed=123)
    y_test  = toy1_function(X_test)
    return X_train, y_train, X_test, y_test

def toy2_function(X):
    x, y, t, z = X[:, 0], X[:, 1], X[:, 2], X[:, 3]
    fx = np.exp(-0.3 * x)
    fy = (0.15 * y) ** 2
    ft = np.tanh(0.3 * t)
    fz = 0.2 * np.sin(0.5 * z + 2) + 0.5
    return fx * fy * ft * fz

def generate_toy2():
    X_train = latin_hypercube_sample(500,  4, low=0.0, high=4.0,  seed=42)
    y_train = toy2_function(X_train)
    X_test  = latin_hypercube_sample(5000, 4, low=0.0, high=10.0, seed=123)
    y_test  = toy2_function(X_test)
    return X_train, y_train, X_test, y_test

if __name__ == "__main__":
    X_tr1, y_tr1, X_te1, y_te1 = generate_toy1()
    X_tr2, y_tr2, X_te2, y_te2 = generate_toy2()
    print("Toy1 train:", X_tr1.shape, y_tr1.shape)
    print("Toy1 test :", X_te1.shape, y_te1.shape)
    print("Toy2 train:", X_tr2.shape, y_tr2.shape)
    print("Toy2 test :", X_te2.shape, y_te2.shape)

Writing dataset_generation.py


In [ ]:
!python dataset_generation.py

Toy1 train: (500, 4) (500,)
Toy1 test : (5000, 4) (5000,)
Toy2 train: (500, 4) (500,)
Toy2 test : (5000, 4) (5000,)


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os
os.makedirs('/content/drive/MyDrive/ISNN_Assignment', exist_ok=True)
print("✓ Ready")

Mounted at /content/drive
✓ Ready


In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from scipy.stats import qmc

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using: {DEVICE}")

# ═══════════════════════════════════════════════════════════
# DATASET GENERATION
# ═══════════════════════════════════════════════════════════
def latin_hypercube_sample(n_samples, n_dims, low, high, seed=42):
    sampler = qmc.LatinHypercube(d=n_dims, seed=seed)
    sample  = sampler.random(n=n_samples)
    return qmc.scale(sample, low, high)

def toy1_function(X):
    x, y, t, z = X[:,0], X[:,1], X[:,2], X[:,3]
    return np.exp(-0.5*x) + np.log(1+np.exp(0.4*y)) + np.tanh(t) + np.sin(z) - 0.4

def toy2_function(X):
    x, y, t, z = X[:,0], X[:,1], X[:,2], X[:,3]
    return np.exp(-0.3*x) * (0.15*y)**2 * np.tanh(0.3*t) * (0.2*np.sin(0.5*z+2)+0.5)

def generate_toy1():
    X_train = latin_hypercube_sample(500,  4, 0.0, 4.0, seed=42)
    X_test  = latin_hypercube_sample(5000, 4, 0.0, 6.0, seed=123)
    return X_train, toy1_function(X_train), X_test, toy1_function(X_test)

def generate_toy2():
    X_train = latin_hypercube_sample(500,  4, 0.0, 4.0,  seed=42)
    X_test  = latin_hypercube_sample(5000, 4, 0.0, 10.0, seed=123)
    return X_train, toy2_function(X_train), X_test, toy2_function(X_test)

# ═══════════════════════════════════════════════════════════
# ACTIVATION FUNCTIONS
# numerically stable softplus using torch.nn.functional
# ═══════════════════════════════════════════════════════════
def sigma_mc(x):
    # Numerically stable softplus = log(1 + exp(x))
    return torch.nn.functional.softplus(x)

def sigma_m(x):
    return torch.sigmoid(x)

def sigma_a(x):
    return torch.sigmoid(x)

# ═══════════════════════════════════════════════════════════
# CONSTRAINED LINEAR LAYER
# Uses F.softplus for stable non-negative weight projection
# ═══════════════════════════════════════════════════════════
class ConstrainedLinear(nn.Module):
    def __init__(self, in_features, out_features, non_negative=True):
        super().__init__()
        self.non_negative = non_negative
        self.weight_raw   = nn.Parameter(
            torch.nn.init.xavier_uniform_(
                torch.empty(out_features, in_features)))
        self.bias = nn.Parameter(torch.zeros(out_features))

    @property
    def weight(self):
        if self.non_negative:
            # Stable softplus keeps weights positive without overflow
            return torch.nn.functional.softplus(self.weight_raw)
        return self.weight_raw

    def forward(self, x):
        return torch.nn.functional.linear(x, self.weight, self.bias)

# ═══════════════════════════════════════════════════════════
# ISNN-1
# ═══════════════════════════════════════════════════════════
class ISNN1(nn.Module):
    def __init__(self, hidden=10, n_layers=2):
        super().__init__()

        # y-branch: σ_mc, non-negative weights
        self.y_layers = nn.ModuleList()
        in_y = 1
        for _ in range(n_layers):
            self.y_layers.append(ConstrainedLinear(in_y, hidden, True))
            in_y = hidden

        # z-branch: σ_a, unconstrained weights
        self.z_layers = nn.ModuleList()
        in_z = 1
        for _ in range(n_layers):
            self.z_layers.append(ConstrainedLinear(in_z, hidden, False))
            in_z = hidden

        # t-branch: σ_m, non-negative weights
        self.t_layers = nn.ModuleList()
        in_t = 1
        for _ in range(n_layers):
            self.t_layers.append(ConstrainedLinear(in_t, hidden, True))
            in_t = hidden

        # x first layer F0 (Eq.4)
        self.Wxx0 = ConstrainedLinear(1,      hidden, False)
        self.Wxy  = ConstrainedLinear(hidden, hidden, True)
        self.Wxz  = ConstrainedLinear(hidden, hidden, False)
        self.Wxt  = ConstrainedLinear(hidden, hidden, True)
        self.bx0  = nn.Parameter(torch.zeros(hidden))

        # x remaining layers (Eq.5)
        self.x_layers = nn.ModuleList()
        for _ in range(n_layers - 1):
            self.x_layers.append(ConstrainedLinear(hidden, hidden, True))

        self.output_layer = nn.Linear(hidden, 1)

    def forward(self, inp):
        x0 = inp[:,0:1]; y0 = inp[:,1:2]
        t0 = inp[:,2:3]; z0 = inp[:,3:4]

        yh = y0
        for layer in self.y_layers:
            yh = sigma_mc(layer(yh))

        zh = z0
        for layer in self.z_layers:
            zh = sigma_a(layer(zh))

        th = t0
        for layer in self.t_layers:
            th = sigma_m(layer(th))

        xh = sigma_mc(
            self.Wxx0(x0) + self.Wxy(yh) +
            self.Wxz(zh)  + self.Wxt(th) + self.bx0
        )

        for layer in self.x_layers:
            xh = sigma_mc(layer(xh))

        return self.output_layer(xh).squeeze(1)

# ═══════════════════════════════════════════════════════════
# ISNN-2
# ═══════════════════════════════════════════════════════════
class ISNN2(nn.Module):
    def __init__(self, hidden=15, n_layers=1):
        super().__init__()

        # y-branch
        self.y_layers = nn.ModuleList()
        in_y = 1
        for _ in range(n_layers):
            self.y_layers.append(ConstrainedLinear(in_y, hidden, True))
            in_y = hidden

        # z-branch
        self.z_layers = nn.ModuleList()
        in_z = 1
        for _ in range(n_layers):
            self.z_layers.append(ConstrainedLinear(in_z, hidden, False))
            in_z = hidden

        # t-branch
        self.t_layers = nn.ModuleList()
        in_t = 1
        for _ in range(n_layers):
            self.t_layers.append(ConstrainedLinear(in_t, hidden, True))
            in_t = hidden

        # x first layer F0 (Eq.9)
        self.F0_x = ConstrainedLinear(1, hidden, False)
        self.F0_y = ConstrainedLinear(1, hidden, True)
        self.F0_z = ConstrainedLinear(1, hidden, False)
        self.F0_t = ConstrainedLinear(1, hidden, True)
        self.F0_b = nn.Parameter(torch.zeros(hidden))

        # x subsequent layers (Eq.10)
        self.Fh_xx   = nn.ModuleList()
        self.Fh_xx0  = nn.ModuleList()
        self.Fh_xy   = nn.ModuleList()
        self.Fh_xz   = nn.ModuleList()
        self.Fh_xt   = nn.ModuleList()
        self.Fh_bias = nn.ParameterList()
        for _ in range(n_layers - 1):
            self.Fh_xx  .append(ConstrainedLinear(hidden, hidden, True))
            self.Fh_xx0 .append(ConstrainedLinear(1,      hidden, False))
            self.Fh_xy  .append(ConstrainedLinear(hidden, hidden, True))
            self.Fh_xz  .append(ConstrainedLinear(hidden, hidden, False))
            self.Fh_xt  .append(ConstrainedLinear(hidden, hidden, True))
            self.Fh_bias.append(nn.Parameter(torch.zeros(hidden)))

        self.output_layer = nn.Linear(hidden, 1)

    def forward(self, inp):
        x0 = inp[:,0:1]; y0 = inp[:,1:2]
        t0 = inp[:,2:3]; z0 = inp[:,3:4]

        yh = y0; y_states = [yh]
        for layer in self.y_layers:
            yh = sigma_mc(layer(yh))
            y_states.append(yh)

        zh = z0; z_states = [zh]
        for layer in self.z_layers:
            zh = sigma_a(layer(zh))
            z_states.append(zh)

        th = t0; t_states = [th]
        for layer in self.t_layers:
            th = sigma_m(layer(th))
            t_states.append(th)

        xh = sigma_mc(
            self.F0_x(x0) + self.F0_y(y0) +
            self.F0_z(z0) + self.F0_t(t0) + self.F0_b
        )

        for i in range(len(self.Fh_xx)):
            xh = sigma_mc(
                self.Fh_xx [i](xh)            +
                self.Fh_xx0[i](x0)            +
                self.Fh_xy [i](y_states[i+1]) +
                self.Fh_xz [i](z_states[i+1]) +
                self.Fh_xt [i](t_states[i+1]) +
                self.Fh_bias[i]
            )

        return self.output_layer(xh).squeeze(1)

# ═══════════════════════════════════════════════════════════
# FFNN
# ═══════════════════════════════════════════════════════════
class FFNN(nn.Module):
    def __init__(self, input_dim=4, hidden=30, n_layers=2):
        super().__init__()
        layers = [nn.Linear(input_dim, hidden), nn.Sigmoid()]
        for _ in range(n_layers - 1):
            layers += [nn.Linear(hidden, hidden), nn.Sigmoid()]
        layers += [nn.Linear(hidden, 1)]
        self.net = nn.Sequential(*layers)

    def forward(self, x):
        return self.net(x).squeeze(1)

# ═══════════════════════════════════════════════════════════
# WEIGHT RESET HELPER
# ═══════════════════════════════════════════════════════════
def reset_weights(model):
    for m in model.modules():
        if isinstance(m, ConstrainedLinear):
            nn.init.xavier_uniform_(m.weight_raw)
            nn.init.zeros_(m.bias)
        elif isinstance(m, nn.Linear):
            nn.init.xavier_uniform_(m.weight)
            nn.init.zeros_(m.bias)

# ═══════════════════════════════════════════════════════════
# TRAINING LOOP
# ═══════════════════════════════════════════════════════════
def train_model(model, X_train, y_train, X_test, y_test,
                n_epochs=30000, lr=1e-3, n_runs=10, log_every=2000):

    all_train, all_test = [], []

    X_tr = torch.tensor(X_train, dtype=torch.float32).to(DEVICE)
    y_tr = torch.tensor(y_train, dtype=torch.float32).to(DEVICE)
    X_te = torch.tensor(X_test,  dtype=torch.float32).to(DEVICE)
    y_te = torch.tensor(y_test,  dtype=torch.float32).to(DEVICE)

    criterion = nn.MSELoss()

    for run in range(n_runs):
        # Fresh weights each run
        reset_weights(model)
        optimizer = optim.Adam(model.parameters(), lr=lr)
        tr_losses, te_losses = [], []
        te_loss_val = 0.0

        for epoch in range(n_epochs):
            model.train()
            optimizer.zero_grad()
            pred = model(X_tr)

            # Safety check — skip update if nan appears
            if torch.isnan(pred).any():
                print(f"  WARNING: NaN at Run {run+1} Epoch {epoch+1}, resetting...")
                reset_weights(model)
                optimizer = optim.Adam(model.parameters(), lr=lr)
                continue

            loss = criterion(pred, y_tr)
            loss.backward()

            # Gradient clipping prevents overflow
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()

            if epoch % 100 == 0:
                model.eval()
                with torch.no_grad():
                    te_pred = model(X_te)
                    te_loss = criterion(te_pred, y_te)
                    te_loss_val = te_loss.item()
                tr_losses.append(loss.item())
                te_losses.append(te_loss_val)

            if (epoch + 1) % log_every == 0:
                print(f"  Run {run+1}/{n_runs}  "
                      f"Epoch {epoch+1}  "
                      f"Train: {loss.item():.4e}  "
                      f"Test:  {te_loss_val:.4e}")

        all_train.append(tr_losses)
        all_test .append(te_losses)

    return model, np.array(all_train), np.array(all_test)

# ═══════════════════════════════════════════════════════════
# PLOTTING
# ═══════════════════════════════════════════════════════════
def plot_losses(results, title_suffix, filename):
    colors = {'FFNN':'red', 'ISNN-1':'blue', 'ISNN-2':'green'}
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    for name, (train_arr, test_arr) in results.items():
        epochs = np.arange(len(train_arr[0])) * 100
        color  = colors.get(name, 'black')
        for ax, arr, label in zip(axes,
                                   [train_arr, test_arr],
                                   ['Training Loss', 'Test Loss']):
            mean = arr.mean(axis=0)
            std  = arr.std(axis=0)
            ax.semilogy(epochs, mean, color=color, label=name)
            ax.fill_between(epochs,
                            np.maximum(mean - std, 1e-10),
                            mean + std,
                            alpha=0.2, color=color)
            ax.set_xlabel('Epoch')
            ax.set_ylabel('MSE Loss')
            ax.set_title(f'{label} — {title_suffix}')
            ax.legend()
            ax.grid(True, which='both', linestyle='--', alpha=0.4)
    plt.tight_layout()
    plt.savefig(filename, dpi=150)
    plt.show()
    print(f"Saved: {filename}")

def plot_behavior(models_dict, true_fn, title_suffix,
                  filename, train_high=4.0, test_high=6.0):
    diag   = np.linspace(0, test_high, 200)
    X_diag = np.column_stack([diag] * 4)
    y_true = true_fn(X_diag)
    n      = len(models_dict)
    fig, axes = plt.subplots(1, n, figsize=(5 * n, 4))
    if n == 1:
        axes = [axes]
    for ax, (name, model) in zip(axes, models_dict.items()):
        model.eval()
        with torch.no_grad():
            X_t  = torch.tensor(X_diag, dtype=torch.float32).to(DEVICE)
            pred = model(X_t).cpu().numpy()
        ax.plot(diag, y_true, 'k--', label='True response')
        ax.plot(diag[diag <= train_high], pred[diag <= train_high],
                'b-', label='Interpolated response')
        ax.plot(diag[diag >  train_high], pred[diag >  train_high],
                'r-', label='Extrapolated response')
        ax.axvline(train_high, color='gray', linestyle=':', alpha=0.6)
        ax.set_xlabel('x = y = t = z')
        ax.set_ylabel('f')
        ax.set_title(name)
        ax.legend(fontsize=7)
        ax.grid(True, linestyle='--', alpha=0.4)
    fig.suptitle(f'Model Behavior — {title_suffix}', fontsize=13)
    plt.tight_layout()
    plt.savefig(filename, dpi=150)
    plt.show()
    print(f"Saved: {filename}")

# ═══════════════════════════════════════════════════════════
# GENERATE DATASETS
# ═══════════════════════════════════════════════════════════
print("\n" + "="*60)
print("GENERATING DATASETS")
print("="*60)
X_tr1, y_tr1, X_te1, y_te1 = generate_toy1()
X_tr2, y_tr2, X_te2, y_te2 = generate_toy2()
print("Toy1 — train:", X_tr1.shape, " test:", X_te1.shape)
print("Toy2 — train:", X_tr2.shape, " test:", X_te2.shape)

N_EPOCHS = 30000
N_RUNS   = 10

# ═══════════════════════════════════════════════════════════
# TOY PROBLEM 1
# ═══════════════════════════════════════════════════════════
print("\n" + "="*60)
print("TOY PROBLEM 1 — PyTorch GPU Training")
print("="*60)

ffnn1   = FFNN(input_dim=4, hidden=30, n_layers=2).to(DEVICE)
isnn1_1 = ISNN1(hidden=10, n_layers=2).to(DEVICE)
isnn2_1 = ISNN2(hidden=15, n_layers=1).to(DEVICE)

print("\nTraining FFNN on Toy1...")
ffnn1,   tr_ff1,  te_ff1  = train_model(
    ffnn1,   X_tr1, y_tr1, X_te1, y_te1, N_EPOCHS, n_runs=N_RUNS)

print("\nTraining ISNN-1 on Toy1...")
isnn1_1, tr_i1_1, te_i1_1 = train_model(
    isnn1_1, X_tr1, y_tr1, X_te1, y_te1, N_EPOCHS, n_runs=N_RUNS)

print("\nTraining ISNN-2 on Toy1...")
isnn2_1, tr_i2_1, te_i2_1 = train_model(
    isnn2_1, X_tr1, y_tr1, X_te1, y_te1, N_EPOCHS, n_runs=N_RUNS)

plot_losses({
    'FFNN':   (tr_ff1,  te_ff1),
    'ISNN-1': (tr_i1_1, te_i1_1),
    'ISNN-2': (tr_i2_1, te_i2_1),
}, 'Toy Problem 1', 'figure3_toy1_losses.png')

plot_behavior({
    'FFNN':   ffnn1,
    'ISNN-1': isnn1_1,
    'ISNN-2': isnn2_1,
}, toy1_function, 'Toy Problem 1',
   'figure4_toy1_behavior.png', 4.0, 6.0)

# ═══════════════════════════════════════════════════════════
# TOY PROBLEM 2
# ═══════════════════════════════════════════════════════════
print("\n" + "="*60)
print("TOY PROBLEM 2 — PyTorch GPU Training")
print("="*60)

ffnn2   = FFNN(input_dim=4, hidden=30, n_layers=2).to(DEVICE)
isnn1_2 = ISNN1(hidden=10, n_layers=2).to(DEVICE)
isnn2_2 = ISNN2(hidden=15, n_layers=1).to(DEVICE)

print("\nTraining FFNN on Toy2...")
ffnn2,   tr_ff2,  te_ff2  = train_model(
    ffnn2,   X_tr2, y_tr2, X_te2, y_te2, N_EPOCHS, n_runs=N_RUNS)

print("\nTraining ISNN-1 on Toy2...")
isnn1_2, tr_i1_2, te_i1_2 = train_model(
    isnn1_2, X_tr2, y_tr2, X_te2, y_te2, N_EPOCHS, n_runs=N_RUNS)

print("\nTraining ISNN-2 on Toy2...")
isnn2_2, tr_i2_2, te_i2_2 = train_model(
    isnn2_2, X_tr2, y_tr2, X_te2, y_te2, N_EPOCHS, n_runs=N_RUNS)

plot_losses({
    'FFNN':   (tr_ff2,  te_ff2),
    'ISNN-1': (tr_i1_2, te_i1_2),
    'ISNN-2': (tr_i2_2, te_i2_2),
}, 'Toy Problem 2', 'figure5_toy2_losses.png')

plot_behavior({
    'FFNN':   ffnn2,
    'ISNN-1': isnn1_2,
    'ISNN-2': isnn2_2,
}, toy2_function, 'Toy Problem 2',
   'figure6_toy2_behavior.png', 4.0, 10.0)

print("\n" + "="*60)
print("ALL DONE — Plots saved!")
print("="*60)

# AUTO SAVE TO DRIVE immediately after training
import shutil
for f in ['figure3_toy1_losses.png',
          'figure4_toy1_behavior.png',
          'figure5_toy2_losses.png',
          'figure6_toy2_behavior.png']:
    if os.path.exists(f):
        shutil.copy(f, f'/content/drive/MyDrive/ISNN_Assignment/{f}')
        print(f"✓ Backed up to Drive: {f}")

# ALSO download directly
from google.colab import files
for f in ['figure3_toy1_losses.png',
          'figure4_toy1_behavior.png',
          'figure5_toy2_losses.png',
          'figure6_toy2_behavior.png']:
    files.download(f)
    print(f"✓ Downloaded: {f}")

Using: cpu

GENERATING DATASETS
Toy1 — train: (500, 4)  test: (5000, 4)
Toy2 — train: (500, 4)  test: (5000, 4)

TOY PROBLEM 1 — PyTorch GPU Training

Training FFNN on Toy1...
  Run 1/10  Epoch 2000  Train: 1.0835e-03  Test:  1.8242e-01
  Run 1/10  Epoch 4000  Train: 1.3506e-04  Test:  1.6651e-01
  Run 1/10  Epoch 6000  Train: 4.8073e-05  Test:  1.5302e-01
  Run 1/10  Epoch 8000  Train: 2.7216e-05  Test:  1.5138e-01
  Run 1/10  Epoch 10000  Train: 1.9509e-05  Test:  1.5194e-01
  Run 1/10  Epoch 12000  Train: 1.5784e-05  Test:  1.4869e-01
  Run 1/10  Epoch 14000  Train: 1.3451e-05  Test:  1.4666e-01
  Run 1/10  Epoch 16000  Train: 1.1808e-05  Test:  1.4644e-01
  Run 1/10  Epoch 18000  Train: 1.0543e-05  Test:  1.4336e-01
  Run 1/10  Epoch 20000  Train: 9.5353e-06  Test:  1.4226e-01
  Run 1/10  Epoch 22000  Train: 8.7146e-06  Test:  1.4105e-01
  Run 1/10  Epoch 24000  Train: 8.0321e-06  Test:  1.4052e-01
  Run 1/10  Epoch 26000  Train: 1.6943e-05  Test:  1.3851e-01
  Run 1/10  Epoch 2800

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

✓ Downloaded: figure3_toy1_losses.png


<IPython.core.display.Javascript object>

...

<IPython.core.display.Javascript object>

✓ Downloaded: figure5_toy2_losses.png


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

✓ Downloaded: figure6_toy2_behavior.png


In [1]:
from google.colab import drive
drive.mount('/content/drive')
import os
os.makedirs('/content/drive/MyDrive/ISNN_Assignment', exist_ok=True)
print("✓ Drive ready")

Mounted at /content/drive
✓ Drive ready


In [2]:
import numpy as np
from scipy.stats import qmc

def latin_hypercube_sample(n_samples, n_dims, low, high, seed=42):
    sampler = qmc.LatinHypercube(d=n_dims, seed=seed)
    sample  = sampler.random(n=n_samples)
    return qmc.scale(sample, low, high)

def toy1_function(X):
    x, y, t, z = X[:,0], X[:,1], X[:,2], X[:,3]
    return np.exp(-0.5*x) + np.log(1+np.exp(0.4*y)) + np.tanh(t) + np.sin(z) - 0.4

def toy2_function(X):
    x, y, t, z = X[:,0], X[:,1], X[:,2], X[:,3]
    return np.exp(-0.3*x) * (0.15*y)**2 * np.tanh(0.3*t) * (0.2*np.sin(0.5*z+2)+0.5)

def generate_toy1():
    X_train = latin_hypercube_sample(500,  4, 0.0, 4.0, seed=42)
    X_test  = latin_hypercube_sample(5000, 4, 0.0, 6.0, seed=123)
    return X_train, toy1_function(X_train), X_test, toy1_function(X_test)

def generate_toy2():
    X_train = latin_hypercube_sample(500,  4, 0.0, 4.0,  seed=42)
    X_test  = latin_hypercube_sample(5000, 4, 0.0, 10.0, seed=123)
    return X_train, toy2_function(X_train), X_test, toy2_function(X_test)

X_tr1, y_tr1, X_te1, y_te1 = generate_toy1()
X_tr2, y_tr2, X_te2, y_te2 = generate_toy2()
print("✓ Toy1 train:", X_tr1.shape, " test:", X_te1.shape)
print("✓ Toy2 train:", X_tr2.shape, " test:", X_te2.shape)

✓ Toy1 train: (500, 4)  test: (5000, 4)
✓ Toy2 train: (500, 4)  test: (5000, 4)


In [3]:
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

# ═══════════════════════════════════════════════════════════
# ACTIVATION FUNCTIONS + DERIVATIVES
# ═══════════════════════════════════════════════════════════
def softplus(x):
    # Numerically stable softplus
    return np.where(x > 30, x, np.log1p(np.exp(np.clip(x, -500, 30))))

def softplus_deriv(x):
    return 1.0 / (1.0 + np.exp(-np.clip(x, -500, 500)))

def sigmoid(x):
    return 1.0 / (1.0 + np.exp(-np.clip(x, -500, 500)))

def sigmoid_deriv(x):
    s = sigmoid(x)
    return s * (1.0 - s)

# ═══════════════════════════════════════════════════════════
# CONSTRAINED WEIGHT HELPERS
# ═══════════════════════════════════════════════════════════
def effective_weight(W_raw, non_negative):
    return softplus(W_raw) if non_negative else W_raw

def effective_weight_grad(W_raw, grad_W_eff, non_negative):
    if non_negative:
        return grad_W_eff * softplus_deriv(W_raw)
    return grad_W_eff

# ═══════════════════════════════════════════════════════════
# NUMPY ISNN-1
# ═══════════════════════════════════════════════════════════
class NumpyISNN1:
    def __init__(self, hidden=10, n_layers=2, lr=1e-3):
        self.hidden   = hidden
        self.n_layers = n_layers
        self.lr       = lr
        self._init_params()

    def _init_params(self):
        H  = self.hidden
        nl = self.n_layers

        def _w(r, c):
            # Xavier initialisation
            limit = np.sqrt(6.0 / (r + c))
            return np.random.uniform(-limit, limit, (r, c))
        def _b(c): return np.zeros(c)

        self.y_W  = [_w(1, H)] + [_w(H, H) for _ in range(nl-1)]
        self.y_b  = [_b(H) for _ in range(nl)]
        self.y_nn = [True] * nl

        self.z_W  = [_w(1, H)] + [_w(H, H) for _ in range(nl-1)]
        self.z_b  = [_b(H) for _ in range(nl)]
        self.z_nn = [False] * nl

        self.t_W  = [_w(1, H)] + [_w(H, H) for _ in range(nl-1)]
        self.t_b  = [_b(H) for _ in range(nl)]
        self.t_nn = [True] * nl

        self.Wxx0_raw = _w(1, H);    self.Wxx0_nn = False
        self.Wxy_raw  = _w(H, H);    self.Wxy_nn  = True
        self.Wxz_raw  = _w(H, H);    self.Wxz_nn  = False
        self.Wxt_raw  = _w(H, H);    self.Wxt_nn  = True
        self.bx0      = _b(H)

        self.x_W  = [_w(H, H) for _ in range(nl-1)]
        self.x_b  = [_b(H)    for _ in range(nl-1)]
        self.x_nn = [True]    * (nl-1)

        self.out_W = _w(H, 1)
        self.out_b = _b(1)

    def forward(self, inp):
        x0 = inp[:,0:1]; y0 = inp[:,1:2]
        t0 = inp[:,2:3]; z0 = inp[:,3:4]
        cache = {'x0':x0,'y0':y0,'t0':t0,'z0':z0}

        # y trunk
        y_pre, y_post = [], []
        yh = y0
        for l in range(self.n_layers):
            W = effective_weight(self.y_W[l], self.y_nn[l])
            z = yh @ W + self.y_b[l]
            a = softplus(z)
            y_pre.append(z); y_post.append(a); yh = a
        cache['y_pre']=y_pre; cache['y_post']=y_post; cache['yH']=yh

        # z trunk
        z_pre, z_post = [], []
        zh = z0
        for l in range(self.n_layers):
            W = effective_weight(self.z_W[l], self.z_nn[l])
            z = zh @ W + self.z_b[l]
            a = sigmoid(z)
            z_pre.append(z); z_post.append(a); zh = a
        cache['z_pre']=z_pre; cache['z_post']=z_post; cache['zH']=zh

        # t trunk
        t_pre, t_post = [], []
        th = t0
        for l in range(self.n_layers):
            W = effective_weight(self.t_W[l], self.t_nn[l])
            z = th @ W + self.t_b[l]
            a = softplus(z)
            t_pre.append(z); t_post.append(a); th = a
        cache['t_pre']=t_pre; cache['t_post']=t_post; cache['tH']=th

        # x first layer F0
        Wxx0 = effective_weight(self.Wxx0_raw, self.Wxx0_nn)
        Wxy  = effective_weight(self.Wxy_raw,  self.Wxy_nn)
        Wxz  = effective_weight(self.Wxz_raw,  self.Wxz_nn)
        Wxt  = effective_weight(self.Wxt_raw,  self.Wxt_nn)
        F0   = x0@Wxx0 + yh@Wxy + zh@Wxz + th@Wxt + self.bx0
        x1   = softplus(F0)
        cache['F0']=F0; cache['x1']=x1

        # x remaining layers
        x_pre, x_post = [F0], [x1]
        xh = x1
        for l in range(len(self.x_W)):
            W = effective_weight(self.x_W[l], self.x_nn[l])
            z = xh @ W + self.x_b[l]
            a = softplus(z)
            x_pre.append(z); x_post.append(a); xh = a
        cache['x_pre']=x_pre; cache['x_post']=x_post; cache['xH']=xh

        out = xh @ self.out_W + self.out_b
        cache['out'] = out
        return out.squeeze(1), cache

    def backward(self, cache, y_true):
        batch = y_true.shape[0]
        out   = cache['out'].squeeze(1)
        delta = (2.0 / batch) * (out - y_true)
        delta = delta[:, np.newaxis]
        grads = {}

        # Output layer
        grads['out_W'] = cache['xH'].T @ delta
        grads['out_b'] = delta.sum(axis=0)
        delta = delta @ self.out_W.T

        # x remaining layers — reversed
        grads['x_W_raw'] = [None]*len(self.x_W)
        grads['x_b']     = [None]*len(self.x_b)
        x_post = cache['x_post']
        for l in reversed(range(len(self.x_W))):
            dact  = softplus_deriv(cache['x_pre'][l+1])
            delta = delta * dact
            W_eff = effective_weight(self.x_W[l], self.x_nn[l])
            grads['x_W_raw'][l] = effective_weight_grad(
                self.x_W[l], x_post[l].T @ delta, self.x_nn[l])
            grads['x_b'][l] = delta.sum(axis=0)
            delta = delta @ W_eff.T

        # x first layer F0
        dact  = softplus_deriv(cache['F0'])
        delta = delta * dact
        Wxy_eff = effective_weight(self.Wxy_raw, self.Wxy_nn)
        Wxz_eff = effective_weight(self.Wxz_raw, self.Wxz_nn)
        Wxt_eff = effective_weight(self.Wxt_raw, self.Wxt_nn)

        grads['Wxx0_raw'] = effective_weight_grad(
            self.Wxx0_raw, cache['x0'].T @ delta, self.Wxx0_nn)
        grads['Wxy_raw']  = effective_weight_grad(
            self.Wxy_raw, cache['yH'].T @ delta, self.Wxy_nn)
        grads['Wxz_raw']  = effective_weight_grad(
            self.Wxz_raw, cache['zH'].T @ delta, self.Wxz_nn)
        grads['Wxt_raw']  = effective_weight_grad(
            self.Wxt_raw, cache['tH'].T @ delta, self.Wxt_nn)
        grads['bx0'] = delta.sum(axis=0)

        delta_yH = delta @ Wxy_eff.T
        delta_zH = delta @ Wxz_eff.T
        delta_tH = delta @ Wxt_eff.T

        # t trunk — reversed
        grads['t_W_raw'] = [None]*self.n_layers
        grads['t_b']     = [None]*self.n_layers
        delta_t  = delta_tH
        t_inputs = [cache['t0']] + cache['t_post'][:-1]
        for l in reversed(range(self.n_layers)):
            dact    = softplus_deriv(cache['t_pre'][l])
            delta_t = delta_t * dact
            W_eff   = effective_weight(self.t_W[l], self.t_nn[l])
            grads['t_W_raw'][l] = effective_weight_grad(
                self.t_W[l], t_inputs[l].T @ delta_t, self.t_nn[l])
            grads['t_b'][l] = delta_t.sum(axis=0)
            delta_t = delta_t @ W_eff.T

        # z trunk — reversed
        grads['z_W_raw'] = [None]*self.n_layers
        grads['z_b']     = [None]*self.n_layers
        delta_z  = delta_zH
        z_inputs = [cache['z0']] + cache['z_post'][:-1]
        for l in reversed(range(self.n_layers)):
            dact    = sigmoid_deriv(cache['z_pre'][l])
            delta_z = delta_z * dact
            W_eff   = effective_weight(self.z_W[l], self.z_nn[l])
            grads['z_W_raw'][l] = effective_weight_grad(
                self.z_W[l], z_inputs[l].T @ delta_z, self.z_nn[l])
            grads['z_b'][l] = delta_z.sum(axis=0)
            delta_z = delta_z @ W_eff.T

        # y trunk — reversed
        grads['y_W_raw'] = [None]*self.n_layers
        grads['y_b']     = [None]*self.n_layers
        delta_y  = delta_yH
        y_inputs = [cache['y0']] + cache['y_post'][:-1]
        for l in reversed(range(self.n_layers)):
            dact    = softplus_deriv(cache['y_pre'][l])
            delta_y = delta_y * dact
            W_eff   = effective_weight(self.y_W[l], self.y_nn[l])
            grads['y_W_raw'][l] = effective_weight_grad(
                self.y_W[l], y_inputs[l].T @ delta_y, self.y_nn[l])
            grads['y_b'][l] = delta_y.sum(axis=0)
            delta_y = delta_y @ W_eff.T

        return grads

    def _init_adam(self):
        self._m = {}; self._v = {}; self._t = 0

    def _adam_step(self, key, param, grad,
                   lr=1e-3, b1=0.9, b2=0.999, eps=1e-8):
        if key not in self._m:
            self._m[key] = np.zeros_like(param)
            self._v[key] = np.zeros_like(param)
        self._m[key] = b1*self._m[key] + (1-b1)*grad
        self._v[key] = b2*self._v[key] + (1-b2)*grad**2
        mh = self._m[key] / (1 - b1**self._t)
        vh = self._v[key] / (1 - b2**self._t)
        return param - lr * mh / (np.sqrt(vh) + eps)

    def update(self, grads):
        self._t += 1; lr = self.lr
        for l in range(self.n_layers):
            self.y_W[l] = self._adam_step(f'yW{l}',self.y_W[l],grads['y_W_raw'][l],lr)
            self.y_b[l] = self._adam_step(f'yb{l}',self.y_b[l],grads['y_b'][l],lr)
            self.z_W[l] = self._adam_step(f'zW{l}',self.z_W[l],grads['z_W_raw'][l],lr)
            self.z_b[l] = self._adam_step(f'zb{l}',self.z_b[l],grads['z_b'][l],lr)
            self.t_W[l] = self._adam_step(f'tW{l}',self.t_W[l],grads['t_W_raw'][l],lr)
            self.t_b[l] = self._adam_step(f'tb{l}',self.t_b[l],grads['t_b'][l],lr)
        self.Wxx0_raw = self._adam_step('Wxx0',self.Wxx0_raw,grads['Wxx0_raw'],lr)
        self.Wxy_raw  = self._adam_step('Wxy', self.Wxy_raw, grads['Wxy_raw'], lr)
        self.Wxz_raw  = self._adam_step('Wxz', self.Wxz_raw, grads['Wxz_raw'], lr)
        self.Wxt_raw  = self._adam_step('Wxt', self.Wxt_raw, grads['Wxt_raw'], lr)
        self.bx0      = self._adam_step('bx0', self.bx0,     grads['bx0'],     lr)
        for l in range(len(self.x_W)):
            self.x_W[l] = self._adam_step(f'xW{l}',self.x_W[l],grads['x_W_raw'][l],lr)
            self.x_b[l] = self._adam_step(f'xb{l}',self.x_b[l],grads['x_b'][l],lr)
        self.out_W = self._adam_step('oW',self.out_W,grads['out_W'],lr)
        self.out_b = self._adam_step('ob',self.out_b,grads['out_b'],lr)

    def mse(self, pred, true):
        return float(np.mean((pred - true)**2))

    def predict(self, X):
        pred, _ = self.forward(X)
        return pred

# ═══════════════════════════════════════════════════════════
# NUMPY ISNN-2
# ═══════════════════════════════════════════════════════════
class NumpyISNN2:
    def __init__(self, hidden=15, n_layers=1, lr=1e-3):
        self.hidden   = hidden
        self.n_layers = n_layers
        self.lr       = lr
        self._init_params()

    def _init_params(self):
        H  = self.hidden
        nl = self.n_layers

        def _w(r, c):
            limit = np.sqrt(6.0 / (r + c))
            return np.random.uniform(-limit, limit, (r, c))
        def _b(c): return np.zeros(c)

        self.y_W  = [_w(1,H)] + [_w(H,H) for _ in range(nl-1)]
        self.y_b  = [_b(H) for _ in range(nl)]
        self.y_nn = [True]*nl

        self.z_W  = [_w(1,H)] + [_w(H,H) for _ in range(nl-1)]
        self.z_b  = [_b(H) for _ in range(nl)]
        self.z_nn = [False]*nl

        self.t_W  = [_w(1,H)] + [_w(H,H) for _ in range(nl-1)]
        self.t_b  = [_b(H) for _ in range(nl)]
        self.t_nn = [True]*nl

        self.F0_x_raw = _w(1,H); self.F0_x_nn = False
        self.F0_y_raw = _w(1,H); self.F0_y_nn = True
        self.F0_z_raw = _w(1,H); self.F0_z_nn = False
        self.F0_t_raw = _w(1,H); self.F0_t_nn = True
        self.F0_b     = _b(H)

        self.Fh_xx_raw  = [_w(H,H) for _ in range(nl-1)]
        self.Fh_xx0_raw = [_w(1,H) for _ in range(nl-1)]
        self.Fh_xy_raw  = [_w(H,H) for _ in range(nl-1)]
        self.Fh_xz_raw  = [_w(H,H) for _ in range(nl-1)]
        self.Fh_xt_raw  = [_w(H,H) for _ in range(nl-1)]
        self.Fh_b       = [_b(H)   for _ in range(nl-1)]
        self.Fh_nn = {'xx':True,'xx0':False,'xy':True,'xz':False,'xt':True}

        self.out_W = _w(H,1)
        self.out_b = _b(1)

    def forward(self, inp):
        x0=inp[:,0:1]; y0=inp[:,1:2]
        t0=inp[:,2:3]; z0=inp[:,3:4]
        cache={'x0':x0,'y0':y0,'t0':t0,'z0':z0}

        y_pre,y_post=[],[y0]
        yh=y0
        for l in range(self.n_layers):
            W=effective_weight(self.y_W[l],self.y_nn[l])
            z=yh@W+self.y_b[l]; a=softplus(z)
            y_pre.append(z); y_post.append(a); yh=a
        cache['y_pre']=y_pre; cache['y_post']=y_post

        z_pre,z_post=[],[z0]
        zh=z0
        for l in range(self.n_layers):
            W=effective_weight(self.z_W[l],self.z_nn[l])
            z=zh@W+self.z_b[l]; a=sigmoid(z)
            z_pre.append(z); z_post.append(a); zh=a
        cache['z_pre']=z_pre; cache['z_post']=z_post

        t_pre,t_post=[],[t0]
        th=t0
        for l in range(self.n_layers):
            W=effective_weight(self.t_W[l],self.t_nn[l])
            z=th@W+self.t_b[l]; a=softplus(z)
            t_pre.append(z); t_post.append(a); th=a
        cache['t_pre']=t_pre; cache['t_post']=t_post

        F0=(x0@effective_weight(self.F0_x_raw,self.F0_x_nn)
           +y0@effective_weight(self.F0_y_raw,self.F0_y_nn)
           +z0@effective_weight(self.F0_z_raw,self.F0_z_nn)
           +t0@effective_weight(self.F0_t_raw,self.F0_t_nn)
           +self.F0_b)
        xh=softplus(F0)
        x_pre=[F0]; x_post=[xh]

        for l in range(len(self.Fh_xx_raw)):
            F=(xh @effective_weight(self.Fh_xx_raw[l], self.Fh_nn['xx'])
              +x0 @effective_weight(self.Fh_xx0_raw[l],self.Fh_nn['xx0'])
              +y_post[l+1]@effective_weight(self.Fh_xy_raw[l], self.Fh_nn['xy'])
              +z_post[l+1]@effective_weight(self.Fh_xz_raw[l], self.Fh_nn['xz'])
              +t_post[l+1]@effective_weight(self.Fh_xt_raw[l], self.Fh_nn['xt'])
              +self.Fh_b[l])
            xh=softplus(F)
            x_pre.append(F); x_post.append(xh)

        cache['x_pre']=x_pre; cache['x_post']=x_post; cache['xH']=xh
        out=xh@self.out_W+self.out_b
        cache['out']=out
        return out.squeeze(1), cache

    def backward(self, cache, y_true):
        batch=y_true.shape[0]
        out=cache['out'].squeeze(1)
        delta=(2.0/batch)*(out-y_true)
        delta=delta[:,np.newaxis]
        grads={}

        grads['out_W']=cache['xH'].T@delta
        grads['out_b']=delta.sum(axis=0)
        delta=delta@self.out_W.T

        grads['Fh_xx_raw'] =[None]*len(self.Fh_xx_raw)
        grads['Fh_xx0_raw']=[None]*len(self.Fh_xx0_raw)
        grads['Fh_xy_raw'] =[None]*len(self.Fh_xy_raw)
        grads['Fh_xz_raw'] =[None]*len(self.Fh_xz_raw)
        grads['Fh_xt_raw'] =[None]*len(self.Fh_xt_raw)
        grads['Fh_b']      =[None]*len(self.Fh_b)

        dy_skip=[np.zeros_like(cache['y_post'][0])]*(self.n_layers+1)
        dz_skip=[np.zeros_like(cache['z_post'][0])]*(self.n_layers+1)
        dt_skip=[np.zeros_like(cache['t_post'][0])]*(self.n_layers+1)

        for l in reversed(range(len(self.Fh_xx_raw))):
            dact=softplus_deriv(cache['x_pre'][l+1])
            delta=delta*dact
            W_xx =effective_weight(self.Fh_xx_raw[l], self.Fh_nn['xx'])
            W_xx0=effective_weight(self.Fh_xx0_raw[l],self.Fh_nn['xx0'])
            W_xy =effective_weight(self.Fh_xy_raw[l], self.Fh_nn['xy'])
            W_xz =effective_weight(self.Fh_xz_raw[l], self.Fh_nn['xz'])
            W_xt =effective_weight(self.Fh_xt_raw[l], self.Fh_nn['xt'])
            grads['Fh_xx_raw'][l] =effective_weight_grad(self.Fh_xx_raw[l], cache['x_post'][l].T@delta,    self.Fh_nn['xx'])
            grads['Fh_xx0_raw'][l]=effective_weight_grad(self.Fh_xx0_raw[l],cache['x0'].T@delta,           self.Fh_nn['xx0'])
            grads['Fh_xy_raw'][l] =effective_weight_grad(self.Fh_xy_raw[l], cache['y_post'][l+1].T@delta,  self.Fh_nn['xy'])
            grads['Fh_xz_raw'][l] =effective_weight_grad(self.Fh_xz_raw[l], cache['z_post'][l+1].T@delta,  self.Fh_nn['xz'])
            grads['Fh_xt_raw'][l] =effective_weight_grad(self.Fh_xt_raw[l], cache['t_post'][l+1].T@delta,  self.Fh_nn['xt'])
            grads['Fh_b'][l]=delta.sum(axis=0)
            dy_skip[l+1]=dy_skip[l+1]+delta@W_xy.T
            dz_skip[l+1]=dz_skip[l+1]+delta@W_xz.T
            dt_skip[l+1]=dt_skip[l+1]+delta@W_xt.T
            delta=delta@W_xx.T

        dact=softplus_deriv(cache['x_pre'][0])
        delta=delta*dact
        F0_yW=effective_weight(self.F0_y_raw,self.F0_y_nn)
        F0_zW=effective_weight(self.F0_z_raw,self.F0_z_nn)
        F0_tW=effective_weight(self.F0_t_raw,self.F0_t_nn)
        grads['F0_x_raw']=effective_weight_grad(self.F0_x_raw,cache['x0'].T@delta,self.F0_x_nn)
        grads['F0_y_raw']=effective_weight_grad(self.F0_y_raw,cache['y0'].T@delta,self.F0_y_nn)
        grads['F0_z_raw']=effective_weight_grad(self.F0_z_raw,cache['z0'].T@delta,self.F0_z_nn)
        grads['F0_t_raw']=effective_weight_grad(self.F0_t_raw,cache['t0'].T@delta,self.F0_t_nn)
        grads['F0_b']=delta.sum(axis=0)
        dy_skip[1]=dy_skip[1]+delta@F0_yW.T
        dz_skip[1]=dz_skip[1]+delta@F0_zW.T
        dt_skip[1]=dt_skip[1]+delta@F0_tW.T

        grads['y_W_raw']=[None]*self.n_layers
        grads['y_b']    =[None]*self.n_layers
        delta_y=dy_skip[self.n_layers]
        for l in reversed(range(self.n_layers)):
            dact=softplus_deriv(cache['y_pre'][l])
            delta_y=delta_y*dact
            W_eff=effective_weight(self.y_W[l],self.y_nn[l])
            grads['y_W_raw'][l]=effective_weight_grad(self.y_W[l],cache['y_post'][l].T@delta_y,self.y_nn[l])
            grads['y_b'][l]=delta_y.sum(axis=0)
            delta_y=delta_y@W_eff.T
            if l>0: delta_y=delta_y+dy_skip[l]

        grads['z_W_raw']=[None]*self.n_layers
        grads['z_b']    =[None]*self.n_layers
        delta_z=dz_skip[self.n_layers]
        for l in reversed(range(self.n_layers)):
            dact=sigmoid_deriv(cache['z_pre'][l])
            delta_z=delta_z*dact
            W_eff=effective_weight(self.z_W[l],self.z_nn[l])
            grads['z_W_raw'][l]=effective_weight_grad(self.z_W[l],cache['z_post'][l].T@delta_z,self.z_nn[l])
            grads['z_b'][l]=delta_z.sum(axis=0)
            delta_z=delta_z@W_eff.T
            if l>0: delta_z=delta_z+dz_skip[l]

        grads['t_W_raw']=[None]*self.n_layers
        grads['t_b']    =[None]*self.n_layers
        delta_t=dt_skip[self.n_layers]
        for l in reversed(range(self.n_layers)):
            dact=softplus_deriv(cache['t_pre'][l])
            delta_t=delta_t*dact
            W_eff=effective_weight(self.t_W[l],self.t_nn[l])
            grads['t_W_raw'][l]=effective_weight_grad(self.t_W[l],cache['t_post'][l].T@delta_t,self.t_nn[l])
            grads['t_b'][l]=delta_t.sum(axis=0)
            delta_t=delta_t@W_eff.T
            if l>0: delta_t=delta_t+dt_skip[l]

        return grads

    def _init_adam(self):
        self._m={}; self._v={}; self._t=0

    def _adam_step(self,key,param,grad,lr=1e-3,b1=0.9,b2=0.999,eps=1e-8):
        if key not in self._m:
            self._m[key]=np.zeros_like(param)
            self._v[key]=np.zeros_like(param)
        self._m[key]=b1*self._m[key]+(1-b1)*grad
        self._v[key]=b2*self._v[key]+(1-b2)*grad**2
        mh=self._m[key]/(1-b1**self._t)
        vh=self._v[key]/(1-b2**self._t)
        return param - lr*mh/(np.sqrt(vh)+eps)

    def update(self, grads):
        self._t+=1; lr=self.lr
        for l in range(self.n_layers):
            self.y_W[l]=self._adam_step(f'yW{l}',self.y_W[l],grads['y_W_raw'][l],lr)
            self.y_b[l]=self._adam_step(f'yb{l}',self.y_b[l],grads['y_b'][l],lr)
            self.z_W[l]=self._adam_step(f'zW{l}',self.z_W[l],grads['z_W_raw'][l],lr)
            self.z_b[l]=self._adam_step(f'zb{l}',self.z_b[l],grads['z_b'][l],lr)
            self.t_W[l]=self._adam_step(f'tW{l}',self.t_W[l],grads['t_W_raw'][l],lr)
            self.t_b[l]=self._adam_step(f'tb{l}',self.t_b[l],grads['t_b'][l],lr)
        self.F0_x_raw=self._adam_step('F0x',self.F0_x_raw,grads['F0_x_raw'],lr)
        self.F0_y_raw=self._adam_step('F0y',self.F0_y_raw,grads['F0_y_raw'],lr)
        self.F0_z_raw=self._adam_step('F0z',self.F0_z_raw,grads['F0_z_raw'],lr)
        self.F0_t_raw=self._adam_step('F0t',self.F0_t_raw,grads['F0_t_raw'],lr)
        self.F0_b    =self._adam_step('F0b',self.F0_b,    grads['F0_b'],    lr)
        for l in range(len(self.Fh_xx_raw)):
            self.Fh_xx_raw[l] =self._adam_step(f'Fxx{l}', self.Fh_xx_raw[l], grads['Fh_xx_raw'][l], lr)
            self.Fh_xx0_raw[l]=self._adam_step(f'Fxx0{l}',self.Fh_xx0_raw[l],grads['Fh_xx0_raw'][l],lr)
            self.Fh_xy_raw[l] =self._adam_step(f'Fxy{l}', self.Fh_xy_raw[l], grads['Fh_xy_raw'][l], lr)
            self.Fh_xz_raw[l] =self._adam_step(f'Fxz{l}', self.Fh_xz_raw[l], grads['Fh_xz_raw'][l], lr)
            self.Fh_xt_raw[l] =self._adam_step(f'Fxt{l}', self.Fh_xt_raw[l], grads['Fh_xt_raw'][l], lr)
            self.Fh_b[l]      =self._adam_step(f'Fb{l}',  self.Fh_b[l],      grads['Fh_b'][l],      lr)
        self.out_W=self._adam_step('oW',self.out_W,grads['out_W'],lr)
        self.out_b=self._adam_step('ob',self.out_b,grads['out_b'],lr)

    def mse(self,pred,true): return float(np.mean((pred-true)**2))
    def predict(self,X):
        pred,_=self.forward(X); return pred

# ═══════════════════════════════════════════════════════════
# NUMPY TRAINING LOOP
# ═══════════════════════════════════════════════════════════
def train_numpy(ModelClass, kwargs, X_train, y_train, X_test, y_test,
                n_epochs=30000, n_runs=10, log_every=2000):
    all_train, all_test = [], []
    for run in range(n_runs):
        np.random.seed(run * 7 + 13)
        model = ModelClass(**kwargs)
        model._init_adam()
        tr_losses, te_losses = [], []
        for epoch in range(n_epochs):
            pred_tr, cache = model.forward(X_train)
            # Skip if nan
            if np.isnan(pred_tr).any():
                print(f"  NaN at Run {run+1} Epoch {epoch+1}, reinitialising...")
                model._init_params()
                model._init_adam()
                continue
            loss_tr = model.mse(pred_tr, y_train)
            grads   = model.backward(cache, y_train)
            model.update(grads)
            pred_te = model.predict(X_test)
            loss_te = model.mse(pred_te, y_test)
            if epoch % 100 == 0:
                tr_losses.append(loss_tr)
                te_losses.append(loss_te)
            if (epoch+1) % log_every == 0:
                print(f"  Run {run+1}/{n_runs}  "
                      f"Epoch {epoch+1}  "
                      f"Train: {loss_tr:.4e}  "
                      f"Test:  {loss_te:.4e}")
        all_train.append(tr_losses)
        all_test .append(te_losses)
    return np.array(all_train), np.array(all_test)

# ═══════════════════════════════════════════════════════════
# NUMPY PLOTTING
# ═══════════════════════════════════════════════════════════
def plot_numpy_losses(results, title_suffix, filename):
    colors = {'ISNN-1 (NumPy)':'blue', 'ISNN-2 (NumPy)':'green'}
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    for name, (train_arr, test_arr) in results.items():
        epochs = np.arange(len(train_arr[0])) * 100
        color  = colors.get(name, 'black')
        for ax, arr, label in zip(axes,
                                   [train_arr, test_arr],
                                   ['Training Loss', 'Test Loss']):
            mean = arr.mean(axis=0)
            std  = arr.std(axis=0)
            ax.semilogy(epochs, mean, color=color, label=name)
            ax.fill_between(epochs,
                            np.maximum(mean-std, 1e-10),
                            mean+std, alpha=0.2, color=color)
            ax.set_xlabel('Epoch')
            ax.set_ylabel('MSE Loss')
            ax.set_title(f'{label} — {title_suffix}')
            ax.legend()
            ax.grid(True, which='both', linestyle='--', alpha=0.4)
    plt.tight_layout()
    plt.savefig(filename, dpi=150)
    plt.show()
    print(f"Saved: {filename}")

def plot_numpy_behavior(models, names, true_fn, title,
                        filename, train_high=4.0, test_high=6.0):
    diag   = np.linspace(0, test_high, 200)
    X_diag = np.column_stack([diag]*4)
    y_true = true_fn(X_diag)
    fig, axes = plt.subplots(1, len(models), figsize=(5*len(models), 4))
    if len(models) == 1: axes = [axes]
    for ax, model, name in zip(axes, models, names):
        pred = model.predict(X_diag)
        ax.plot(diag, y_true, 'k--', label='True response')
        ax.plot(diag[diag<=train_high], pred[diag<=train_high],
                'b-', label='Interpolated response')
        ax.plot(diag[diag>train_high],  pred[diag>train_high],
                'r-', label='Extrapolated response')
        ax.axvline(train_high, color='gray', linestyle=':', alpha=0.6)
        ax.set_xlabel('x = y = t = z')
        ax.set_ylabel('f')
        ax.set_title(name)
        ax.legend(fontsize=7)
        ax.grid(True, linestyle='--', alpha=0.4)
    fig.suptitle(title, fontsize=13)
    plt.tight_layout()
    plt.savefig(filename, dpi=150)
    plt.show()
    print(f"Saved: {filename}")

# ═══════════════════════════════════════════════════════════
# NUMPY MAIN
# ═══════════════════════════════════════════════════════════
N_EPOCHS = 30000
N_RUNS   = 10

print("\n" + "="*60)
print("NUMPY — TOY PROBLEM 1")
print("="*60)

tr_i1_1, te_i1_1 = train_numpy(
    NumpyISNN1, dict(hidden=10, n_layers=2, lr=1e-3),
    X_tr1, y_tr1, X_te1, y_te1, N_EPOCHS, N_RUNS)

tr_i2_1, te_i2_1 = train_numpy(
    NumpyISNN2, dict(hidden=15, n_layers=1, lr=1e-3),
    X_tr1, y_tr1, X_te1, y_te1, N_EPOCHS, N_RUNS)

plot_numpy_losses({
    'ISNN-1 (NumPy)': (tr_i1_1, te_i1_1),
    'ISNN-2 (NumPy)': (tr_i2_1, te_i2_1),
}, 'Toy Problem 1', 'numpy_figure3_toy1_losses.png')

# Train single model for behavior plot
np.random.seed(0)
m1_toy1 = NumpyISNN1(hidden=10, n_layers=2, lr=1e-3); m1_toy1._init_adam()
m2_toy1 = NumpyISNN2(hidden=15, n_layers=1, lr=1e-3); m2_toy1._init_adam()
print("\nTraining single models for Toy1 behavior plot...")
for ep in range(N_EPOCHS):
    p,c=m1_toy1.forward(X_tr1); g=m1_toy1.backward(c,y_tr1); m1_toy1.update(g)
    p,c=m2_toy1.forward(X_tr1); g=m2_toy1.backward(c,y_tr1); m2_toy1.update(g)
    if (ep+1) % 5000 == 0: print(f"  Behavior model epoch {ep+1}/{N_EPOCHS}")

plot_numpy_behavior(
    [m1_toy1, m2_toy1],
    ['ISNN-1 (NumPy)', 'ISNN-2 (NumPy)'],
    toy1_function, 'NumPy Behavior — Toy Problem 1',
    'numpy_figure4_toy1_behavior.png', 4.0, 6.0)

print("\n" + "="*60)
print("NUMPY — TOY PROBLEM 2")
print("="*60)

tr_i1_2, te_i1_2 = train_numpy(
    NumpyISNN1, dict(hidden=10, n_layers=2, lr=1e-3),
    X_tr2, y_tr2, X_te2, y_te2, N_EPOCHS, N_RUNS)

tr_i2_2, te_i2_2 = train_numpy(
    NumpyISNN2, dict(hidden=15, n_layers=1, lr=1e-3),
    X_tr2, y_tr2, X_te2, y_te2, N_EPOCHS, N_RUNS)

plot_numpy_losses({
    'ISNN-1 (NumPy)': (tr_i1_2, te_i1_2),
    'ISNN-2 (NumPy)': (tr_i2_2, te_i2_2),
}, 'Toy Problem 2', 'numpy_figure5_toy2_losses.png')

np.random.seed(0)
m1_toy2 = NumpyISNN1(hidden=10, n_layers=2, lr=1e-3); m1_toy2._init_adam()
m2_toy2 = NumpyISNN2(hidden=15, n_layers=1, lr=1e-3); m2_toy2._init_adam()
print("\nTraining single models for Toy2 behavior plot...")
for ep in range(N_EPOCHS):
    p,c=m1_toy2.forward(X_tr2); g=m1_toy2.backward(c,y_tr2); m1_toy2.update(g)
    p,c=m2_toy2.forward(X_tr2); g=m2_toy2.backward(c,y_tr2); m2_toy2.update(g)
    if (ep+1) % 5000 == 0: print(f"  Behavior model epoch {ep+1}/{N_EPOCHS}")

plot_numpy_behavior(
    [m1_toy2, m2_toy2],
    ['ISNN-1 (NumPy)', 'ISNN-2 (NumPy)'],
    toy2_function, 'NumPy Behavior — Toy Problem 2',
    'numpy_figure6_toy2_behavior.png', 4.0, 10.0)

print("\n" + "="*60)
print("ALL NUMPY DONE — Plots saved!")
print("="*60)

# ADD THESE LINES AT THE VERY END OF THE NUMPY CELL

# Save to Google Drive
import shutil
numpy_plots = [
    'numpy_figure3_toy1_losses.png',
    'numpy_figure4_toy1_behavior.png',
    'numpy_figure5_toy2_losses.png',
    'numpy_figure6_toy2_behavior.png',
]
for f in numpy_plots:
    if os.path.exists(f):
        shutil.copy(f, f'/content/drive/MyDrive/ISNN_Assignment/{f}')
        print(f"✓ Saved to Drive: {f}")

# Download to PC
from google.colab import files
for f in numpy_plots:
    if os.path.exists(f):
        files.download(f)
        print(f"✓ Downloaded: {f}")


NUMPY — TOY PROBLEM 1
  Run 1/10  Epoch 2000  Train: 7.5977e-01  Test:  3.4710e+00
  Run 1/10  Epoch 4000  Train: 7.0453e-01  Test:  3.2416e+00
  Run 1/10  Epoch 6000  Train: 6.3570e-01  Test:  2.8992e+00
  Run 1/10  Epoch 8000  Train: 5.2699e-01  Test:  2.3351e+00
  Run 1/10  Epoch 10000  Train: 3.6339e-01  Test:  1.4264e+00
  Run 1/10  Epoch 12000  Train: 2.0561e-01  Test:  4.8658e-01
  Run 1/10  Epoch 14000  Train: 5.8269e+00  Test:  1.5716e+01
  Run 1/10  Epoch 16000  Train: 1.0375e-01  Test:  1.6358e-01
  Run 1/10  Epoch 18000  Train: 8.0722e-02  Test:  1.4337e-01
  Run 1/10  Epoch 20000  Train: 5.3365e-02  Test:  1.2241e-01
  Run 1/10  Epoch 22000  Train: 3.6471e-02  Test:  1.1140e-01
  Run 1/10  Epoch 24000  Train: 3.1969e-02  Test:  1.1206e-01
  Run 1/10  Epoch 26000  Train: 3.0688e-02  Test:  1.1625e-01
  Run 1/10  Epoch 28000  Train: 8.2182e-02  Test:  1.3888e-01
  Run 1/10  Epoch 30000  Train: 3.0241e-02  Test:  1.2851e-01
  Run 2/10  Epoch 2000  Train: 8.0332e-01  Test:  2

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

✓ Downloaded: numpy_figure3_toy1_losses.png


<IPython.core.display.Javascript object>

...

<IPython.core.display.Javascript object>

✓ Downloaded: numpy_figure5_toy2_losses.png


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

✓ Downloaded: numpy_figure6_toy2_behavior.png


In [3]:
import numpy as np
import pandas as pd
from scipy.stats import qmc
from google.colab import files

# Regenerate datasets
def latin_hypercube_sample(n_samples, n_dims, low, high, seed=42):
    sampler = qmc.LatinHypercube(d=n_dims, seed=seed)
    sample  = sampler.random(n=n_samples)
    return qmc.scale(sample, low, high)

def toy1_function(X):
    x, y, t, z = X[:,0], X[:,1], X[:,2], X[:,3]
    return np.exp(-0.5*x) + np.log(1+np.exp(0.4*y)) + np.tanh(t) + np.sin(z) - 0.4

def toy2_function(X):
    x, y, t, z = X[:,0], X[:,1], X[:,2], X[:,3]
    return np.exp(-0.3*x) * (0.15*y)**2 * np.tanh(0.3*t) * (0.2*np.sin(0.5*z+2)+0.5)

X_tr1 = latin_hypercube_sample(500,  4, 0.0, 4.0,  seed=42)
X_te1 = latin_hypercube_sample(5000, 4, 0.0, 6.0,  seed=123)
y_tr1 = toy1_function(X_tr1)
y_te1 = toy1_function(X_te1)

X_tr2 = latin_hypercube_sample(500,  4, 0.0, 4.0,  seed=42)
X_te2 = latin_hypercube_sample(5000, 4, 0.0, 10.0, seed=123)
y_tr2 = toy2_function(X_tr2)
y_te2 = toy2_function(X_te2)

print("✓ Datasets regenerated")
print("Toy1 train:", X_tr1.shape, " test:", X_te1.shape)
print("Toy2 train:", X_tr2.shape, " test:", X_te2.shape)

# Save as CSV
cols = ['x', 'y', 't', 'z', 'output']
pd.DataFrame(np.column_stack([X_tr1, y_tr1]), columns=cols).to_csv('toy1_train.csv', index=False)
pd.DataFrame(np.column_stack([X_te1, y_te1]), columns=cols).to_csv('toy1_test.csv',  index=False)
pd.DataFrame(np.column_stack([X_tr2, y_tr2]), columns=cols).to_csv('toy2_train.csv', index=False)
pd.DataFrame(np.column_stack([X_te2, y_te2]), columns=cols).to_csv('toy2_test.csv',  index=False)
print("✓ CSV files created")

# Download
for f in ['toy1_train.csv','toy1_test.csv','toy2_train.csv','toy2_test.csv']:
    files.download(f)
    print(f"✓ Downloaded: {f}")

✓ Datasets regenerated
Toy1 train: (500, 4)  test: (5000, 4)
Toy2 train: (500, 4)  test: (5000, 4)
✓ CSV files created


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

✓ Downloaded: toy1_train.csv


<IPython.core.display.Javascript object>

...

<IPython.core.display.Javascript object>

✓ Downloaded: toy2_train.csv


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

✓ Downloaded: toy2_test.csv
